# Phase 2: prepare ISIC 2019 images and check for hidden duplicates

**Decision-support prototype only. Not a diagnosis. Consult a qualified clinician.**

No training here. This notebook (1) downloads the official ISIC 2019 training images (9.8 GB, no Kaggle needed),
(2) resizes only the NEW training images to 300x300 and saves them to your Google Drive (so a disconnect never costs a re-download),
and (3) looks for near-duplicate pictures between the new training images and your frozen HAM10000 validation/test images.
Image ids and lesion ids were already checked offline; this step catches the same photo hiding under a different id.
Run the 3 code cells in order. Total time: about 25-40 minutes, mostly the download. A GPU is not needed.

Data source: ISIC 2019 Challenge training data (BCN_20000, HAM10000, MSK). Licence and attribution are in the zip (LICENSE.txt, ATTRIBUTION.txt): read them.

## Cell 1 of 3: setup and download
Mounts Drive, asks you to upload 3 small CSV files from `docs/results/` (`split2_train.csv`, `split_val.csv`, `split_test.csv`), downloads and unzips the images.

In [ ]:
import os, glob, json, time, subprocess
import numpy as np, pandas as pd
from PIL import Image
from google.colab import drive, files
drive.mount("/content/drive")

# ---------------- ONE CONFIG BLOCK ----------------
CFG = dict(
    zip_url="https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip",
    input_size=300,                      # same as the model contract: resize straight to 300x300, no crop
    drive_out="/content/drive/MyDrive/skin_capstone/isic2019_300",
    hash_size=8,                         # dHash: 8x8 = 64 bits
    near_dup_max_bits=10,                # pairs differing in <= this many of 64 bits are listed for review
)
os.makedirs(CFG["drive_out"], exist_ok=True)

need = ["split2_train.csv", "split_val.csv", "split_test.csv"]
if not all(os.path.exists(f"/content/{n}") for n in need):
    up = files.upload()                  # select the 3 files
    for n in need: open(f"/content/{n}", "wb").write(up[n])
train2, val, test = (pd.read_csv(f"/content/{n}") for n in need)
new_imgs = train2[train2["source"] != "HAM10000"]
print("new training images to prepare:", len(new_imgs), "| frozen val+test:", len(val) + len(test))

if not os.path.exists("/content/isic/ISIC_2019_Training_Input"):
    subprocess.run(f"wget -q -O /content/isic.zip {CFG['zip_url']} && mkdir -p /content/isic && unzip -q /content/isic.zip -d /content/isic && rm /content/isic.zip", shell=True, check=True)
index = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob("/content/isic/**/*.jpg", recursive=True)}
print("images found in zip:", len(index))
assert len(index) == 25331, "unexpected image count: check the downloaded ISIC 2019 files"
assert all(i in index for i in pd.concat([new_imgs, val, test])["image_id"]), "some needed images are missing"
for lic in glob.glob("/content/isic/**/LICENSE.txt", recursive=True) + glob.glob("/content/isic/**/ATTRIBUTION.txt", recursive=True):
    print("=====", lic); print(open(lic).read()[:1500])

## Cell 2 of 3: resize new images to Drive and fingerprint images
Resizing uses the exact torchvision transform of the model contract, so pixels match what the backend would produce. A 64-bit "difference hash" (dHash) is computed for each image: two copies of the same photo get almost the same hash.
Re-running is safe: already-saved images are skipped.

In [ ]:
from torchvision import transforms
resize = transforms.Resize((CFG["input_size"], CFG["input_size"]))

def dhash(img: Image.Image, n: int = CFG["hash_size"]) -> int:
    """Tiny fingerprint: compare each pixel with its right neighbour on an (n+1) x n grayscale thumbnail."""
    g = np.asarray(img.convert("L").resize((n + 1, n), Image.LANCZOS), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

hashes, t0 = {}, time.time()
new_set = set(new_imgs["image_id"])
all_ids = list(new_imgs["image_id"]) + list(val["image_id"]) + list(test["image_id"])
for k, iid in enumerate(all_ids):
    img = Image.open(index[iid]).convert("RGB")
    hashes[iid] = dhash(img)
    out = f"{CFG['drive_out']}/{iid}.png"
    if iid in new_set and not os.path.exists(out):
        resize(img).save(out)                                     # lossless PNG, 300x300
    if k % 2000 == 0: print(k, f"{time.time()-t0:.0f}s")
print("saved to Drive:", len(glob.glob(CFG["drive_out"] + "/*.png")))

## Cell 3 of 3: list near-duplicates between new training images and frozen val/test, then download the report
`dup_candidates.csv` lists every (new training image, val/test image) pair whose fingerprints differ in at most 10 of 64 bits (distance 0 = identical picture).
Send this file back; images that are true duplicates will be removed from training.

In [ ]:
frozen = pd.concat([val.assign(split="val"), test.assign(split="test")])
new_ids = new_imgs["image_id"].tolist(); fz_ids = frozen["image_id"].tolist()
A = np.array([hashes[i] for i in new_ids], dtype=np.uint64); B = np.array([hashes[i] for i in fz_ids], dtype=np.uint64)

def popcount(x: np.ndarray) -> np.ndarray:
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return (x * np.uint64(0x0101010101010101)) >> np.uint64(56)

rows = []
for j in range(0, len(A), 500):                       # chunked so memory stays small
    d = popcount(A[j:j+500, None] ^ B[None, :])
    for a, b in zip(*np.where(d <= CFG["near_dup_max_bits"])):
        rows.append(dict(train_image=new_ids[j + a], frozen_image=fz_ids[b], frozen_split=frozen.iloc[b]["split"], bits_different=int(d[a, b])))
dups = pd.DataFrame(rows).sort_values("bits_different") if rows else pd.DataFrame(columns=["train_image", "frozen_image", "frozen_split", "bits_different"])
dups.to_csv("/content/dup_candidates.csv", index=False)
print("candidate pairs:", len(dups)); print(dups.head(20)); print(dups["bits_different"].value_counts().sort_index().to_dict() if len(dups) else "none")
json.dump(dict(n_new=len(new_ids), n_frozen=len(fz_ids), max_bits=CFG["near_dup_max_bits"], pairs=len(dups)), open("/content/dup_summary.json", "w"))
files.download("/content/dup_candidates.csv"); files.download("/content/dup_summary.json")